<a href="https://colab.research.google.com/github/expaetra/CM3070_final_project/blob/master/05_representations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!git clone https://github.com/expaetra/CM3070_final_project.git
%cd CM3070_final_project

Cloning into 'CM3070_final_project'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 94 (delta 16), reused 74 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 16.19 MiB | 14.56 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/CM3070_final_project


In [10]:
import os
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords

A stopword list is created using standard English stopwords together with domain specific frequent words identified during EDA.

In [8]:
df = pd.read_csv("backend/data/arxiv_abstracts_cleaned.csv")
print(df.shape)
print(df.head())

(15717, 5)
  category             field               discipline  \
0    cs.LG  Machine Learning  Artificial Intelligence   
1    cs.LG  Machine Learning  Artificial Intelligence   
2    cs.LG  Machine Learning  Artificial Intelligence   
3    cs.LG  Machine Learning  Artificial Intelligence   
4    cs.LG  Machine Learning  Artificial Intelligence   

                                            abstract  abstract_length  
0  design rule checking (drc) is getting increasi...              117  
1  anomaly detection is a key application of mach...              125  
2  the increased reliance on the internet and the...              220  
3  clinical coding is an administrative process t...              159  
4  the translation of medical diagnosis to clinic...              154  


In [11]:
nltk.download("stopwords")

# define domain-specific stropwords
custom_stops = {
    'based', 'paper', 'show', 'results', 'problem', 'using', 'approach',
    'proposed', 'method', 'methods', 'propose', 'present', 'work',
    'used', 'use', 'two', 'one', 'new', 'also', 'shows', 'however',
    'provide', 'study', 'task', 'tasks', 'different', 'high', 'given'
}

# join with standard stopwords
stop_words = set(stopwords.words('english')).union(custom_stops)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


After establishing a baseline using TF-IDF and logistic regression this section will explore different text representations while keeping the classifier fixed to the baseline. The goal of this process is to understand how the choice of representation affects model performance.

Any differences in results can be attributed to the representation itself and not the the model. This allows for a fair comparison and helps us to identify which representation captures the most useful information from the text.

- TF-IDF word unigrams
- TF-IDF word n-grams (1,2)
- TF-IDF word n-grams (1,3)
- TF-IDF character n-gram
- Word2Vec-based document vectors

In [12]:
# prepare labels & split data

# extract abstracts & make sure they are strings
X_text = df["abstract"].astype(str)

# labels for disciplines & fields
y_disc = df["discipline"]
y_field = df["field"]

X_train_text, X_test_text, y_disc_train, y_disc_test, y_field_train, y_field_test = train_test_split(
    X_text,
    y_disc,
    y_field,
    test_size=0.2,
    random_state=42,
    stratify=y_disc
)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))

Train size: 12573
Test size: 3144


In [17]:
# evaluation helper function

def evaluate_model(model, X_train, X_test, y_train, y_test,task_name, representation_name):

    # train the model on training data
    model.fit(X_train, y_train)

    # predict labels for test data
    y_pred = model.predict(X_test)

    # calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # calculate macro-averaged precision, recall, F1
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )

    # calculate weighted averaged precision, recall, F1
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted",zero_division=0
    )

    return {
        "Task": task_name,
        "Representation": representation_name,
        "Model": "Logistic Regression",
        "Accuracy": round(accuracy),
        "Macro Precision": round(macro_p, 3),
        "Macro Recall": round(macro_r, 3),
        "Macro F1": round(macro_f1, 3),
        "Weighted Precision": round(weighted_p, 3),
        "Weighted Recall": round(weighted_r, 3),
        "Weighted F1": round(weighted_f1, 3)
    }

results = []

### TF-IDF word unigrams
This is the baseline representation, where each word is treated independently. It captures basic word importance across documents.

In [18]:
# create TF-IDF vectorizer using single words (unigrams)

vectorizer_uni = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1,1),
    max_features=5000
)

# convert training text into tf-idf feature vectors and transform test text
X_train_uni = vectorizer_uni.fit_transform(X_train_text)
X_test_uni = vectorizer_uni.transform(X_test_text)

# train & evaluate model for discipline classification
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_uni, X_test_uni,
        y_disc_train, y_disc_test,
        "Discipline", "TF-IDF word unigrams"
    )
)

# train & evaluate model for field classification
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000,random_state=42),
        X_train_uni, X_test_uni,
        y_field_train, y_field_test,
        "Field", "TF-IDF word unigrams"
    )
)

print("TF-IDF word unigrams done")

TF-IDF word unigrams done


### TF-IDF word n-grams (1-2)
This representation includes single words and two-word phrases, which allows the model to capture short local context.

In [20]:
vectorizer_12 = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    max_features=5000
)

X_train_12 = vectorizer_12.fit_transform(X_train_text)
X_test_12 = vectorizer_12.transform(X_test_text)

# discipline classification
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_12, X_test_12,
        y_disc_train,y_disc_test,
        "Discipline", "TF-IDF word n-grams (1-2)"
    )
)

# field classification
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_12, X_test_12,
        y_field_train, y_field_test,
        "Field", "TF-IDF word n-grams (1-2)"
    )
)

# Print confirmation message
print("TF-IDF word n-grams (1-2) done")

TF-IDF word n-grams Done


### TF-IDF word n-grams (1-3)
Extend the representation to three-word phrases. It may capture more meaning but it can also make the feature space sparser.

In [21]:
vectorizer_13 = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1,3),
    max_features=5000
)

X_train_13 = vectorizer_13.fit_transform(X_train_text)
X_test_13 = vectorizer_13.transform(X_test_text)

# discipline
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_13, X_test_13,
        y_disc_train, y_disc_test,
        "Discipline", "TF-IDF word n-grams (1-3)"
    )
)

# field
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_13,X_test_13,
        y_field_train, y_field_test,
        "Field", "TF-IDF word n-grams (1-3)"
    )
)

print("TF-IDF word n-grams (1-3) done")

TF-IDF word n-grams (1-3) done


### TF-IDF character n-grams
Character n-grams focus on subword patterns instead of full words. This can be useful for capturing frequent term fragments.

In [22]:
vectorizer_char = TfidfVectorizer(

    # use characters instead of words
    analyzer="char",

    # sequence of 3 to 5 characters
    ngram_range=(3, 5),
    max_features=5000
)

X_train_char = vectorizer_char.fit_transform(X_train_text)
X_test_char = vectorizer_char.transform(X_test_text)

# discipline
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000,random_state=42),
        X_train_char, X_test_char,
        y_disc_train, y_disc_test,
        "Discipline", "TF-IDF character n-grams"
    )
)


# field
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_char, X_test_char,
        y_field_train, y_field_test,
        "Field", "TF-IDF character n-grams"
    )
)

print("TF-IDF character n-grams done")

TF-IDF character n-grams done


### Word2Vec document embeddings
Word2Vec is used to learn word embeddings and then document vectors are built by averaging the embeddings of words in each abstract

In [23]:
# tokenize text into words
def tokenize(text):

    # keep words > 3 letters
    words =re.findall(r'\b[a-z]{3,}\b', text.lower())

    # remove stopwords
    return [w for w in words if w not in stop_words]

In [24]:
# apply tokenizaiton on test and training

train_tokens = [tokenize(text) for text in X_train_text]
test_tokens = [tokenize(text) for text in X_test_text]

In [25]:
# train Word2Vec model

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

In [26]:
# create a document vector by averaging word vectors
def document_vector(tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors)==0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

# convert documents into vectors
X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in train_tokens])
X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in test_tokens])

# discipline
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_w2v, X_test_w2v,
        y_disc_train, y_disc_test,
        "Discipline", "Word2Vec document embeddings"
    )
)

# field
results.append(
    evaluate_model(
        LogisticRegression(max_iter=1000, random_state=42),
        X_train_w2v, X_test_w2v,
        y_field_train, y_field_test,
        "Field", "Word2Vec document embeddings"
    )
)

print("Word2Vec document embeddings done")

Word2Vec document embeddings done


### Results

In [27]:
# create a results table

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["Task", "Macro F1"], ascending=[True, False])
print(results_df)
results_df

          Task                Representation                Model  Accuracy  \
0   Discipline          TF-IDF word unigrams  Logistic Regression         1   
2   Discipline     TF-IDF word n-grams (1-2)  Logistic Regression         1   
4   Discipline     TF-IDF word n-grams (1-2)  Logistic Regression         1   
6   Discipline     TF-IDF word n-grams (1-3)  Logistic Regression         1   
8   Discipline      TF-IDF character n-grams  Logistic Regression         1   
10  Discipline  Word2Vec document embeddings  Logistic Regression         0   
1        Field          TF-IDF word unigrams  Logistic Regression         1   
7        Field     TF-IDF word n-grams (1-3)  Logistic Regression         1   
3        Field     TF-IDF word n-grams (1-2)  Logistic Regression         1   
5        Field     TF-IDF word n-grams (1-2)  Logistic Regression         1   
9        Field      TF-IDF character n-grams  Logistic Regression         0   
11       Field  Word2Vec document embeddings  Logist

,Task,Representation,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0,Discipline,TF-IDF word unigrams,Logistic Regression,1,0.575,0.478,0.514,0.562,0.555,0.543
2,Discipline,TF-IDF word n-grams (1-2),Logistic Regression,1,0.577,0.475,0.512,0.563,0.553,0.541
4,Discipline,TF-IDF word n-grams (1-2),Logistic Regression,1,0.577,0.475,0.512,0.563,0.553,0.541
6,Discipline,TF-IDF word n-grams (1-3),Logistic Regression,1,0.577,0.475,0.512,0.563,0.554,0.542
8,Discipline,TF-IDF character n-grams,Logistic Regression,1,0.547,0.446,0.484,0.530,0.521,0.510
10,Discipline,Word2Vec document embeddings,Logistic Regression,0,0.504,0.428,0.453,0.499,0.499,0.486
1,Field,TF-IDF word unigrams,Logistic Regression,1,0.500,0.509,0.500,0.502,0.509,0.501
7,Field,TF-IDF word n-grams (1-3),Logistic Regression,1,0.498,0.506,0.498,0.500,0.506,0.499
3,Field,TF-IDF word n-grams (1-2),Logistic Regression,1,0.495,0.504,0.495,0.497,0.504,0.496
5,Field,TF-IDF word n-grams (1-2),Logistic Regression,1,0.495,0.504,0.495,0.497,0.504,0.496


TF-IDF word unigrams perform best for both discipline and field classification. Adding n-grams did not improve performance and character n-grams and Word2Vec perform worse. This shows that simple word-based TF-IDF captures the most useful information.

In [28]:
import os

os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)

results_df.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/representation_results.csv",
    index=False
)
print("Saved")

Saved
